In [5]:
#!ln -s /rds/general/project/hda_25-26 ~/rds_project

from pathlib import Path
import sys

try:
    _here = Path(__file__).resolve()
except NameError:
    _here = Path.cwd()  

_root = next(p for p in [_here, *_here.parents] if (p / "config.py").exists())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from config import PROJECT_ROOT

from pathlib import Path
import pandas as pd
import numpy as np
import pyreadr
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import itertools
import warnings
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")
from sklearn.utils import resample


In [6]:
# ─────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────

print("=" * 60)
print("LOADING DATA")
print("=" * 60)

# ── Project root ──────────────────────────────────────────────
PROJECT_ROOT   = Path("/rds/general/project/hda_25-26/live/TDS/fg520/TDS-Group-4") # CHANGE ROOT DIRECTORY HERE !

analysis_path  = PROJECT_ROOT / "00_data" / "02_analysis" /"stability_selection"/ "stability_selection_runs_outcome" 
processed_path = PROJECT_ROOT / "00_data" / "01_processed" / "03_imputation"
output_path = PROJECT_ROOT / "02_results" / "03_ml" / "00_neural_network"
nn_out_dir = PROJECT_ROOT /"02_results"/"03_ml"/"nn_outputs" # Separate folder for combined results 
nn_out_dir.mkdir(parents=True, exist_ok=True)

# ── Validate paths exist before loading ───────────────────────
for p in [analysis_path, processed_path]:
    if not p.exists():
        raise FileNotFoundError(f"Path not found: {p}")

# ── Load selected feature names ───────────────────────────────
selected_names_path = analysis_path / "selected_variable_names.csv"
if not selected_names_path.exists():
    raise FileNotFoundError(f"Selected variable names file not found: {selected_names_path}")

selected_names = pd.read_csv(selected_names_path)
feature_names  = selected_names.iloc[:, 0].tolist()
print(f"Stability-selected features loaded: {len(feature_names)}")
print(f"  Features: {feature_names}")

# ── Load RDS files ────────────────────────────────────────────
for rds_file in ["df_train_imputed.rds", "df_test_imputed.rds"]:
    if not (processed_path / rds_file).exists():
        raise FileNotFoundError(f"RDS file not found: {processed_path / rds_file}")

train = pyreadr.read_r(processed_path / "df_train_imputed.rds")[None]
test  = pyreadr.read_r(processed_path / "df_test_imputed.rds")[None]

# ── Basic validation ──────────────────────────────────────────
assert "has_cvd" in train.columns, "'has_cvd' column missing from train set"
assert "has_cvd" in test.columns,  "'has_cvd' column missing from test set"
assert len(train) > 0, "Train set is empty"
assert len(test)  > 0, "Test set is empty"

# ── Check selected features are present ───────────────────────
missing_from_train = [f for f in feature_names if f not in train.columns]
missing_from_test  = [f for f in feature_names if f not in test.columns]
if missing_from_train:
    print(f"  Warning: {len(missing_from_train)} selected features missing from train: {missing_from_train}")
if missing_from_test:
    print(f"  Warning: {len(missing_from_test)} selected features missing from test:  {missing_from_test}")

# ── Summary ───────────────────────────────────────────────────
print(f"\nDataset shapes:  Train {train.shape}  |  Test {test.shape}")
print(f"CVD prevalence:  Train {train['has_cvd'].mean():.3%}  |  Test {test['has_cvd'].mean():.3%}")
print(f"Missing values:  Train {train.isnull().sum().sum()}  |  Test {test.isnull().sum().sum()}")
print(f"\nData types in train:\n{train[feature_names].dtypes.value_counts()}")

LOADING DATA
Stability-selected features loaded: 4
  Features: ['triglycerides', 'crp', 'employment', 'smoking_status_refined']

Dataset shapes:  Train (40000, 51)  |  Test (40000, 51)
CVD prevalence:  Train 10.740%  |  Test 10.808%
Missing values:  Train 73408  |  Test 73299


KeyError: "['smoking_status_refined'] not in index"

In [ ]:
# ─────────────────────────────────────────────
# 2. FEATURE SELECTION & PREPROCESSING
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("PREPROCESSING")
print("=" * 60)

# Intersect with available columns (drop any not present)

feature_names = selected_names.iloc[:, 0].tolist()
# confounders = ['age', 'sex', 'ethnicity']
# feature_names = feature_names + [c for c in confounders if c not in feature_names]

available = [f for f in feature_names if f in train.columns]
missing    = [f for f in feature_names if f not in train.columns]

print(f"Stability-selected features: {len(selected_names)}")
# print(f"Confounders requested:       {confounders}")
print(f"Features available in data:  {len(available)}")
if missing:
    print(f"WARNING - not found in data: {missing}")
print(f"Final feature list: {available}")

TARGET = 'has_cvd'

# Drop rows where target is missing
train = train.dropna(subset=[TARGET])
test  = test.dropna(subset=[TARGET])

X_train = train[available].copy()
y_train = train[TARGET].astype(int)
X_test  = test[available].copy()
y_test  = test[TARGET].astype(int)

# One-hot encode categorical columns
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
if cat_cols:
    X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)
    X_test  = pd.get_dummies(X_test,  columns=cat_cols, drop_first=True)
    # Align columns after encoding
    X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

extra_test_cols = set(X_test.columns) - set(X_train.columns)
if extra_test_cols:
    print(f"Warning: {len(extra_test_cols)} test-only categories dropped: {extra_test_cols}")

# Split train into train + validation BEFORE scaling
X_tr_raw, X_val_raw, y_tr_raw, y_val_raw = train_test_split(
    X_train, y_train, test_size=0.25, stratify=y_train, random_state=42
)

# Fit scaler on sub-train only
scaler = StandardScaler()
X_tr_sc  = scaler.fit_transform(X_tr_raw)
X_val_sc = scaler.transform(X_val_raw)
X_te_sc  = scaler.transform(X_test)

# Tensors
X_tr  = torch.tensor(X_tr_sc,  dtype=torch.float32)
y_tr  = torch.tensor(y_tr_raw.values, dtype=torch.float32)
X_val = torch.tensor(X_val_sc, dtype=torch.float32)
y_val = torch.tensor(y_val_raw.values, dtype=torch.float32)
X_te  = torch.tensor(X_te_sc,  dtype=torch.float32)
y_te  = torch.tensor(y_test.values, dtype=torch.float32)

BATCH_SIZE = 512
train_ds     = TensorDataset(X_tr, y_tr)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"Train: {X_tr.shape[0]}  |  Val: {X_val.shape[0]}  |  Test: {X_te.shape[0]}")

# Compute pos_weight from training labels
n_neg = (y_tr == 0).sum().item()
n_pos = (y_tr == 1).sum().item()
pos_weight_val = torch.tensor([n_neg / n_pos], dtype=torch.float32)
print(f"Class balance — CVD: {n_pos/(n_pos+n_neg):.3%}  |  pos_weight: {pos_weight_val.item():.2f}")

In [ ]:
# ─────────────────────────────────────────────
# 3. MODEL DEFINITION
# ─────────────────────────────────────────────

def build_model(input_dim, hidden_layers, hidden_units, dropout_rate):
    """
    Build an MLP with variable depth and width.

    Parameters
    ----------
    input_dim    : int   – number of input features
    hidden_layers: int   – number of hidden layers
    hidden_units : int   – neurons per hidden layer
    dropout_rate : float – dropout probability (0 = no dropout)
    """
    layers = []
    in_dim = input_dim
    for _ in range(hidden_layers):
        layers.append(nn.Linear(in_dim, hidden_units))
        layers.append(nn.BatchNorm1d(hidden_units))
        layers.append(nn.ReLU())
        if dropout_rate > 0:
            layers.append(nn.Dropout(dropout_rate))
        in_dim = hidden_units
    layers.append(nn.Linear(in_dim, 1))   # binary output
    # Note: no Sigmoid here; BCEWithLogitsLoss handles it
    return nn.Sequential(*layers)

def train_model(model, loader, n_epochs, lr, device, pos_weight=None, 
                X_val=None, y_val=None, patience=10):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    
    pw = pos_weight.to(device) if pos_weight is not None else None
    criterion = nn.BCEWithLogitsLoss(pos_weight=pw)

    epoch_losses = []
    best_val_loss = float('inf')
    patience_counter = 0
    best_state = None

    for epoch in range(n_epochs):
        model.train()
        running_loss = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb).squeeze(1)
            loss = criterion(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            running_loss += loss.item() * len(xb)

        epoch_loss = running_loss / len(loader.dataset)
        epoch_losses.append(epoch_loss)

        if np.isnan(epoch_loss):
            print(f"  NaN loss at epoch {epoch+1}, stopping early")
            break

        # ── Early stopping on validation loss ──────────────────
        if X_val is not None:
            model.eval()
            with torch.no_grad():
                val_logits = model(X_val.to(device)).squeeze(1)
                val_loss   = criterion(val_logits, y_val.to(device)).item()
            if val_loss < best_val_loss:
                best_val_loss    = val_loss
                best_state       = {k: v.clone() for k, v in model.state_dict().items()}
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience:
                    print(f"  Early stop at epoch {epoch+1} (best val loss: {best_val_loss:.4f})")
                    break

    # Restore best weights
    if best_state is not None:
        model.load_state_dict(best_state)

    return epoch_losses

def evaluate_model(model, X, y, device, batch_size=1024):
    """Batched inference — safe for large datasets."""
    model.eval()
    all_probs = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            xb     = X[i : i + batch_size].to(device)
            logits = model(xb).squeeze(1)
            all_probs.append(torch.sigmoid(logits).cpu())
    probs = torch.cat(all_probs).numpy()
    auc   = roc_auc_score(y.numpy(), probs)
    return auc, probs

def bootstrap_roc(probs, labels, n_boot=1000, seed=42):
    """Return mean FPR grid, mean TPR, and 95% CI band via bootstrapping."""
    rng = np.random.RandomState(seed)
    base_fpr = np.linspace(0, 1, 101)
    tprs = []
    for _ in range(n_boot):
        p_b, l_b = resample(probs, labels, random_state=rng)
        # skip degenerate bootstrap samples (only one class)
        if len(np.unique(l_b)) < 2:
            continue
        fpr_b, tpr_b, _ = roc_curve(l_b, p_b)
        tprs.append(np.interp(base_fpr, fpr_b, tpr_b))
    tprs = np.array(tprs)
    return base_fpr, tprs.mean(0), np.percentile(tprs, 2.5, axis=0), np.percentile(tprs, 97.5, axis=0)


In [ ]:
# ─────────────────────────────────────────────
# 4. HYPERPARAMETER GRID SEARCH
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("HYPERPARAMETER TUNING")
print("=" * 60)

INPUT_DIM = X_tr.shape[1]

# Grid 
param_grid = {
    'hidden_layers': [1, 2, 3],       # depth
    'hidden_units':  [32, 64, 128],   # width
    'dropout_rate':  [0.0, 0.3],      # regularisation
    'lr':            [1e-3],          # learning rate
    'n_epochs':      [30],            # fixed for grid search speed
}

# # Quicker grid search, 8 instead of 18 
# param_grid = {
#     'hidden_layers': [1, 2],       # depth
#     'hidden_units':  [32, 64],   # width
#     'dropout_rate':  [0.0, 0.3],      # regularisation
#     'lr':            [1e-3],          # learning rate
#     'n_epochs':      [30],            # fixed for grid search speed
# }

keys   = list(param_grid.keys())
combos = list(itertools.product(*param_grid.values()))
print(f"Total configurations to evaluate: {len(combos)}")

results = []
for i, combo in enumerate(combos):
    params = dict(zip(keys, combo))
    print(f"\n[{i+1}/{len(combos)}] {params}", end="  ")

    torch.manual_seed(42)
    model = build_model(
        INPUT_DIM,
        hidden_layers=params['hidden_layers'],
        hidden_units =params['hidden_units'],
        dropout_rate =params['dropout_rate'],
    )

    train_model(
        model, train_loader, params['n_epochs'], params['lr'], device,
        pos_weight=pos_weight_val,
        X_val=X_val, y_val=y_val, patience=5   # light patience for grid search
    )

    auc_train, _ = evaluate_model(model, X_tr,  y_tr,  device)
    auc_val,   _ = evaluate_model(model, X_val, y_val, device)

    print(f"→ Train AUC={auc_train:.4f}  Val AUC={auc_val:.4f}")

    results.append({**params, 'train_auc': auc_train, 'val_auc': auc_val})

results_df = pd.DataFrame(results).sort_values('val_auc', ascending=False)
print("\nTOP 5 CONFIGURATIONS BY VALIDATION AUC")
print(results_df.head(5).to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────
# 5. RETRAIN BEST MODEL (MORE EPOCHS)
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("RETRAINING BEST MODEL")
print("=" * 60)

best = results_df.iloc[0]
print(f"Best params: layers={int(best.hidden_layers)}, units={int(best.hidden_units)}, "
      f"dropout={best.dropout_rate}, lr={best.lr}")

torch.manual_seed(42)
best_model = build_model(
    INPUT_DIM,
    hidden_layers=int(best.hidden_layers),
    hidden_units =int(best.hidden_units),
    dropout_rate =best.dropout_rate,
)

loss_curve = train_model(
    best_model, train_loader, n_epochs=150, lr=best.lr, device=device,
    pos_weight=pos_weight_val,
    X_val=X_val, y_val=y_val, patience=15   # more patience for final model
)

auc_train_final, prob_train = evaluate_model(best_model, X_tr,  y_tr,  device)
auc_val_final,   prob_val   = evaluate_model(best_model, X_val, y_val, device)
auc_test_final,  prob_test  = evaluate_model(best_model, X_te,  y_te,  device)

auprc = average_precision_score(y_te.numpy(), prob_test)

print(f"Final — Train AUC: {auc_train_final:.4f}  |  "
      f"Val AUC: {auc_val_final:.4f}  |  "
      f"Test AUC: {auc_test_final:.4f}  |  "
      f"Test AUPRC: {auprc:.4f}")

In [ ]:
# ─────────────────────────────────────────────
# 6. EVALUATION & PLOTS
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("EVALUATION")
print("=" * 60)

# Find threshold that maximises F1 on the VALIDATION set
fpr_v, tpr_v, thresholds_v = roc_curve(y_val.numpy(), prob_val)
f1_scores = []
for t in thresholds_v:
    preds = (prob_val >= t).astype(int)
    tp = ((preds == 1) & (y_val.numpy() == 1)).sum()
    fp = ((preds == 1) & (y_val.numpy() == 0)).sum()
    fn = ((preds == 0) & (y_val.numpy() == 1)).sum()
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1_scores.append(2 * prec * rec / (prec + rec + 1e-8))

best_threshold = thresholds_v[np.argmax(f1_scores)]
print(f"Optimal threshold (val F1): {best_threshold:.3f}")

# Apply to test set
y_pred = (prob_test >= best_threshold).astype(int)
print(f"\nClassification Report (threshold={best_threshold:.3f}):")
print(classification_report(y_te.numpy(), y_pred, target_names=['No CVD', 'CVD']))

fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# --- (A) Loss curve
ax0 = fig.add_subplot(gs[0, 0])
ax0.plot(loss_curve, color='steelblue', lw=2)
ax0.set_title('Training Loss Curve', fontweight='bold')
ax0.set_xlabel('Epoch'); ax0.set_ylabel('BCE Loss')
ax0.grid(alpha=0.3)

# --- (B) ROC curve — train vs test with 95% CI
ax1 = fig.add_subplot(gs[0, 1])
for probs, labels, label, col in [
    (prob_train, y_tr.numpy(), f'Train (AUC={auc_train_final:.3f})', 'steelblue'),
    (prob_test,  y_te.numpy(), f'Test  (AUC={auc_test_final:.3f})',  'tomato'),
]:
    # Point-estimate curve
    fpr, tpr, _ = roc_curve(labels, probs)
    ax1.plot(fpr, tpr, label=label, color=col, lw=2)

    # Bootstrapped 95% CI band
    base_fpr, mean_tpr, tpr_lo, tpr_hi = bootstrap_roc(probs, labels)
    ax1.fill_between(base_fpr, tpr_lo, tpr_hi, color=col, alpha=0.15,
                     label=f'  95% CI')

ax1.plot([0, 1], [0, 1], 'k--', lw=1)
ax1.set_title('ROC Curves', fontweight='bold')
ax1.set_xlabel('FPR'); ax1.set_ylabel('TPR')
ax1.legend(fontsize=9); ax1.grid(alpha=0.3)

# --- (C) Predicted probability distribution
ax2 = fig.add_subplot(gs[0, 2])
ax2.hist(prob_test[y_test.values == 0], bins=50, alpha=0.6, color='steelblue', label='No CVD', density=True)
ax2.hist(prob_test[y_test.values == 1], bins=50, alpha=0.6, color='tomato',    label='CVD',    density=True)
ax2.set_title('Predicted Probability Distribution', fontweight='bold')
ax2.set_xlabel('P(CVD)'); ax2.set_ylabel('Density')
ax2.legend(); ax2.grid(alpha=0.3)

# --- (D) Hyperparameter heatmap — AUC by layers × units (dropout=best)
ax3 = fig.add_subplot(gs[1, 0:2])
pivot = (
    results_df[results_df['dropout_rate'] == best.dropout_rate]
    .pivot_table(index='hidden_layers', columns='hidden_units', values='val_auc')
)
im = ax3.imshow(pivot.values, aspect='auto', cmap='YlOrRd',
                vmin=results_df['val_auc'].min(),
                vmax=results_df['val_auc'].max())
ax3.set_xticks(range(len(pivot.columns))); ax3.set_xticklabels(pivot.columns.astype(int))
ax3.set_yticks(range(len(pivot.index)));   ax3.set_yticklabels(pivot.index.astype(int))
ax3.set_xlabel('Hidden Units'); ax3.set_ylabel('Hidden Layers')
ax3.set_title(f'Validation AUC Heatmap (dropout={best.dropout_rate})', fontweight='bold')
plt.colorbar(im, ax=ax3, shrink=0.8)
for (r, c), val in np.ndenumerate(pivot.values):
    ax3.text(c, r, f'{val:.3f}', ha='center', va='center', fontsize=9,
             color='black' if val < pivot.values.max() - 0.01 else 'white')


# --- (E) Confusion matrix
ax4 = fig.add_subplot(gs[1, 2])
cm  = confusion_matrix(y_test.values, y_pred)
im2 = ax4.imshow(cm, cmap='Blues')
ax4.set_xticks([0, 1]); ax4.set_xticklabels(['No CVD', 'CVD'])
ax4.set_yticks([0, 1]); ax4.set_yticklabels(['No CVD', 'CVD'])
ax4.set_xlabel('Predicted'); ax4.set_ylabel('Actual')
ax4.set_title('Confusion Matrix (threshold=0.5)', fontweight='bold')
plt.colorbar(im2, ax=ax4, shrink=0.8)
for (r, c), val in np.ndenumerate(cm):
    ax4.text(c, r, str(val), ha='center', va='center',
             fontsize=12, color='white' if val > cm.max() / 2 else 'black')

fig.suptitle('CVD Neural Network — Evaluation Summary', fontsize=14, fontweight='bold', y=1.01)
plt.savefig(output_path / "nn_cvd_evaluation.png",
            dpi=150, bbox_inches='tight')
plt.show()
print("\nPlot saved.")

In [ ]:
# ─────────────────────────────────────────────
# 7. SAVE RESULTS
# ─────────────────────────────────────────────
results_df.to_csv(
    output_path / "nn_hyperparameter_results.csv",
    index=False
)
print("Hyperparameter results saved.")

# Save best model weights
torch.save(best_model.state_dict(), output_path / "nn_best_model.pt")

print("Best model weights saved.")
print("\nDone.")

pd.DataFrame({
    "y_true": y_te.numpy(),
    "prob_train_model": prob_test,
}).to_csv(output_path / "nn_best_model_test_predictions.csv", index=False)

In [ ]:
# ─────────────────────────────────────────────
# 8. THREE-MODEL COMPARISON
# ─────────────────────────────────────────────

print("\n" + "=" * 60)
print("THREE-MODEL COMPARISON")
print("=" * 60)

TARGET = 'has_cvd'

# ── Define feature sets ───────────────────────────────────────
# Adjust these column names to match your data exactly
exposome_features    = selected_names.iloc[:, 0].tolist()
demographic_features  = ['age', 'sex', 'ethnicity']   # adjust if needed
combined_features     = exposome_features + [d for d in demographic_features 
                                              if d not in exposome_features]

feature_sets = {
    'Exposome Only\n(Stability Selected)': exposome_features,
    'Demographics Only\n(Age, Sex, Ethnicity)': demographic_features,
    'Combined\n(Exposome + Demographics)': combined_features,
}

# ── Helper: prepare data, train, evaluate ─────────────────────
def run_model_pipeline(feature_list, train_df, test_df, target, device,
                       n_epochs=150, patience=15, batch_size=512, seed=42):
    """Full pipeline for a given feature set. Returns test AUC, probs, labels."""
    
    available = [f for f in feature_list if f in train_df.columns]
    missing   = [f for f in feature_list if f not in train_df.columns]
    if missing:
        print(f"  Warning: {len(missing)} features not found: {missing}")

    # Split features / target
    X_tr_raw = train_df[available].copy()
    X_te_raw = test_df[available].copy()
    y_tr_raw = train_df[target].astype(int)
    y_te_raw = test_df[target].astype(int)

    # One-hot encode categoricals
    cat_cols = X_tr_raw.select_dtypes(include=['object', 'category']).columns.tolist()
    if cat_cols:
        X_tr_raw = pd.get_dummies(X_tr_raw, columns=cat_cols, drop_first=True)
        X_te_raw = pd.get_dummies(X_te_raw, columns=cat_cols, drop_first=True)
        X_tr_raw, X_te_raw = X_tr_raw.align(X_te_raw, join='left', axis=1, fill_value=0)

    # Impute
    X_tr_raw = X_tr_raw.fillna(X_tr_raw.median())
    X_te_raw = X_te_raw.fillna(X_tr_raw.median())

    # Val split
    X_sub, X_val_raw, y_sub, y_val_raw = train_test_split(
        X_tr_raw, y_tr_raw, test_size=0.15, stratify=y_tr_raw, random_state=seed
    )

    # Scale
    sc = StandardScaler()
    X_sub_sc = sc.fit_transform(X_sub)
    X_val_sc = sc.transform(X_val_raw)
    X_te_sc  = sc.transform(X_te_raw)

    # Tensors
    X_sub_t = torch.tensor(X_sub_sc,        dtype=torch.float32)
    y_sub_t = torch.tensor(y_sub.values,    dtype=torch.float32)
    X_val_t = torch.tensor(X_val_sc,        dtype=torch.float32)
    y_val_t = torch.tensor(y_val_raw.values, dtype=torch.float32)
    X_te_t  = torch.tensor(X_te_sc,         dtype=torch.float32)
    y_te_t  = torch.tensor(y_te_raw.values, dtype=torch.float32)

    # pos_weight
    n_neg = (y_sub_t == 0).sum().item()
    n_pos = (y_sub_t == 1).sum().item()
    pw    = torch.tensor([n_neg / n_pos], dtype=torch.float32)

    # DataLoader
    loader = DataLoader(TensorDataset(X_sub_t, y_sub_t),
                        batch_size=batch_size, shuffle=True)

    # Build & train model
    torch.manual_seed(seed)
    model = build_model(
        input_dim    = X_sub_t.shape[1],
        hidden_layers= 1,
        hidden_units = 64,
        dropout_rate = 0.3,
    )

    train_model(
        model, loader, n_epochs=n_epochs, lr=1e-3, device=device,
        pos_weight=pw, X_val=X_val_t, y_val=y_val_t, patience=patience
    )

    auc_train, _         = evaluate_model(model, X_sub_t, y_sub_t, device)
    auc_val,   _         = evaluate_model(model, X_val_t, y_val_t, device)
    auc_test,  prob_test = evaluate_model(model, X_te_t,  y_te_t,  device)
    auprc = average_precision_score(y_te_t.numpy(), prob_test)

    return {
        'auc_train': auc_train,
        'auc_val':   auc_val,
        'auc_test':  auc_test,
        'auprc':     auprc,
        'probs':     prob_test,
        'labels':    y_te_t.numpy(),
        'n_features': X_sub_t.shape[1],
    }

# ── Run all three models ───────────────────────────────────────
model_results = {}

for model_name, features in feature_sets.items():
    clean_name = model_name.replace('\n', ' ')
    print(f"\n{'─'*50}")
    print(f"Running: {clean_name}")
    print(f"  Features requested: {len(features)}")

    res = run_model_pipeline(features, train, test, TARGET, device)
    model_results[model_name] = res

    print(f"  Features used (post-encoding): {res['n_features']}")
    print(f"  Train AUC: {res['auc_train']:.4f}")
    print(f"  Val AUC:   {res['auc_val']:.4f}")
    print(f"  Test AUC:  {res['auc_test']:.4f}")
    print(f"  AUPRC:     {res['auprc']:.4f}")

# ── Summary table ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
summary = pd.DataFrame({
    name.replace('\n', ' '): {
        'N Features':  res['n_features'],
        'Train AUC':   round(res['auc_train'], 4),
        'Val AUC':     round(res['auc_val'],   4),
        'Test AUC':    round(res['auc_test'],  4),
        'AUPRC':       round(res['auprc'],     4),
    }
    for name, res in model_results.items()
}).T
print(summary.to_string())

# ── Overlaid ROC plot with 95% CI ─────────────────────────────
colours = ['steelblue', 'darkorange', 'tomato']
styles  = ['--', '-.', '-']

fig, ax = plt.subplots(figsize=(8, 7))

for (model_name, res), colour, style in zip(model_results.items(), colours, styles):
    label_clean = model_name.replace(chr(10), ' ')

    # Point-estimate curve
    fpr, tpr, _ = roc_curve(res['labels'], res['probs'])
    ax.plot(fpr, tpr, color=colour, lw=2.5, linestyle=style,
            label=f"{label_clean}  (AUC={res['auc_test']:.3f})")

    # Bootstrapped 95% CI band
    base_fpr, mean_tpr, tpr_lo, tpr_hi = bootstrap_roc(res['probs'], res['labels'])
    ax.fill_between(base_fpr, tpr_lo, tpr_hi, color=colour, alpha=0.12,
                    label=f"  95% CI")

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random chance (AUC=0.500)')

ax.set_xlabel('False Positive Rate', fontsize=13)
ax.set_ylabel('True Positive Rate', fontsize=13)
ax.set_title('ROC Curves — Three-Model Comparison\nCVD Prediction (UK Biobank)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='lower right', framealpha=0.9)
ax.grid(alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])

# Annotate AUC difference
auc_exposome = model_results[list(feature_sets.keys())[0]]['auc_test']
auc_combined  = model_results[list(feature_sets.keys())[2]]['auc_test']

ax.annotate(
    f"Exposome → Combined: ΔAUC = +{auc_combined - auc_exposome:.3f}",
    xy=(0.4, 0.08), fontsize=10, color='dimgray',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='gray')
)

plt.tight_layout()
plt.savefig(output_path / "nn_roc_comparison.png",
    dpi=150, bbox_inches='tight'
)
plt.show()
print("ROC comparison plot saved.")


# Save summary table
summary.to_csv( output_path / "nn_three_model_comparison.csv")
print("Summary table saved.")


In [24]:
# Save NN test predictions for later combined ROC plotting

from pathlib import Path
import pandas as pd

nn_pred_df = pd.DataFrame({
    "y_true": list(model_results.values())[0]["labels"]  # same test labels for all three models
})

for model_name, res in model_results.items():
    safe_name = (
        model_name.lower()
        .replace("\n", "_")
        .replace("(", "")
        .replace(")", "")
        .replace("+", "plus")
        .replace(",", "")
        .replace(" ", "_")
    )
    nn_pred_df[f"{safe_name}_prob"] = res["probs"]

nn_pred_df.to_csv(nn_out_dir / "nn_test_predictions.csv", index=False)
nn_pred_df.head()

,y_true,exposome_only_stability_selected_prob
0,1.0,0.497025
1,0.0,0.497548
2,0.0,0.489411
3,0.0,0.493710
4,0.0,0.491442
